In [12]:
import os
import re
import pandas as pd
import yaml

# === CONFIG ===
PROJECTS_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\July_31\YAML_Files"
OUTPUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "3.1_YML_List_ShallowC.csv")

# === UNIT TEST KEYWORDS ===
UNIT_TEST_KEYWORDS = [
    'gradlew test', './gradlew test', './gradlew jvmtest',
    'testdebugunittest', 'testreleaseunittest',
    'kotlintest', 'unittest', 'run unit tests',
    'run: test', 'npm test', 'yarn test', 'run: flutter test'
]

# === DEVICE & TEST TRIGGER LABEL SETS ===
DEVICE_SETUP_LABELS = {
    'GitHub_emulator_full', 'GitHub_emulator_compact', 'GitHub_emulator_manual',
    'GitHub_GMD', 'Firebase_Full', 'Firebase_Compact',
    'Appcenter', 'Browserstack', 'SauceLabs'
}
TEST_TRIGGERS = {
    'Gradle_Command_Legacy', 'Firebase_Full', 'Firebase_Compact',
    'Appcenter', 'Browserstack', 'SauceLabs'
}

# === DETECT CI PLATFORM ===
def detect_ci_platform(file_path):
    filename = os.path.basename(file_path)
    if "__" in filename and "++" in filename:
        try:
            return filename.split("__")[1].split("++")[0]
        except Exception:
            return "Other"
    return "Other"

# === DETECT UNIT TEST ===
def detect_unit_test(yaml_text):
    text = yaml_text.lower()
    return any(kw in text for kw in UNIT_TEST_KEYWORDS)

# === EXTRACT RUN SCRIPTS ===
def extract_run_scripts(parsed):
    def get_runs(node):
        runs = []
        if isinstance(node, dict):
            for k, v in node.items():
                if k == 'run' and isinstance(v, str):
                    runs.append(v.lower())
                else:
                    runs.extend(get_runs(v))
        elif isinstance(node, list):
            for item in node:
                runs.extend(get_runs(item))
        return runs

    scripts = get_runs(parsed)
    if isinstance(parsed, dict):
        for key in ['script', 'before_script', 'before_install', 'after_script']:
            val = parsed.get(key)
            if isinstance(val, list):
                scripts.extend([str(v).lower() for v in val])
            elif isinstance(val, str):
                scripts.append(val.lower())
    return scripts

# === INSTRUMENTATION TEST DETECTION ===
def detect_instrumentation_by_platform(ci_platform, yaml_text, run_scripts, parsed):
    run_text = "\n".join(run_scripts)
    found = set()
    matched = []

    # === Emulator Setup ===
    has_sdk = any(kw in run_text for kw in ['sdkmanager', 'android update sdk'])
    has_avd = any(kw in run_text for kw in ['avdmanager', 'android create avd'])

    has_emulator = any(kw in run_text for kw in ['emulator'])
    #has_emulator = 'emulator' in run_text
    has_emulator_wait = 'android-wait-for-emulator' in run_text
    has_adb_keyevent = 'adb shell input keyevent' in run_text


    if ci_platform.lower() == "github_actions":
        for job in parsed.get('jobs', {}).values():
            for step in job.get('steps', []):
                if isinstance(step, dict) and 'uses' in step:
                    uses = step['uses'].lower()
                    if 'reactivecircus/android-emulator-runner' in uses:
                        found.add('GitHub_emulator_full')
                        matched.append('reactivecircus/android-emulator-runner')
                    if 'malinskiy/action-android/emulator-run-cmd' in uses:
                        found.add('GitHub_emulator_compact')
                        matched.append('malinskiy/action-android/emulator-run-cmd')
                    if any(gmd in uses for gmd in ['cleanmanageddevices', 'managedvirtualdevice', 'manageddevices']):
                        found.add('GitHub_GMD')
                        matched.append('GMD keywords (uses)')

        for s in run_scripts:
            if re.search(r'gradlew.*(managed)?device.*test', s):
                found.add('GitHub_GMD')
                matched.append('GMD test run')

        if (has_sdk or has_avd) and (has_emulator or has_avd):
            if any(re.search(r'(connected.*(test|check)|instrumentation|assemble.*androidtest|run.*instrument.*test|orchestrator|install.*debug)', s) for s in run_scripts):
                found.add('GitHub_emulator_manual')
                matched.append('manual emulator + test run')

    # Detect manual emulator setup for non-GitHub platforms (including Travis)
    if (has_sdk or has_avd or has_emulator or has_emulator_wait or has_adb_keyevent):
        if any(re.search(r'(connected.*(test|check)|instrumentation|assemble.*androidtest|run.*instrument.*test|orchestrator|install.*debug)', s) for s in run_scripts):
            found.add(f"{ci_platform}_emulator_manual")
            matched.append('manual emulator + test run')


    # === Gradle Commands ===
    COMMON_INSTR_TEST_CMDS = [
    'connectedandroidtest',       # canonical instrumentation test
    'connectedcheck',             # often used to trigger connected tests
    'assembleandroidtest',        # builds APK for instr tests
    'assembledebugandroidtest',   # more specific variant
    'installdebug',               # install test APK on emulator
    'uninstallall',               # cleanup step for instrumentation
    'am instrument',              # CLI-based instrumentation run
    'connectedflavortest',        # used for flavor-specific connected tests
    'createinstrumentationtestcoveragereport',  # used in coverage pipelines
    'connecteddevicetest',        # alt naming
    'connectedtest',              # often a custom Gradle alias
    'runinstrumentationtests',    # Firebase or wrapper tasks
    'executescreenshottests',     # screenshot instrumentation tests (e.g., Facebook’s screenshot-test)
    'orchestrator'                # commonly used for instrumentation orchestration
]
    for cmd in COMMON_INSTR_TEST_CMDS:
        if any(cmd in s.lower() for s in run_scripts):
            found.add('Gradle_Command_Legacy')
            matched.append(f'gradle_legacy: {cmd}')

    COMMON_EXACT_CMD_PHRASES = [
        './gradlew connectedandroidtest', './gradlew connectedcheck',
        './gradlew assembledebugandroidtest', './gradlew installdebug',
        './gradlew createDebugCoverageReport', './gradlew runinstrumentationtests',
        'gcloud firebase test android run', 'am instrument'
    ]
    for exact in COMMON_EXACT_CMD_PHRASES:
        if any(s.strip().lower() == exact for s in run_scripts):
            found.add('Gradle_Command_Legacy')
            matched.append(f'exact_legacy: {exact}')

    # === Third-party device labs ===
    for job in parsed.get('jobs', {}).values():
        for step in job.get('steps', []):
            if isinstance(step, dict) and 'uses' in step:
                uses = step['uses'].lower()
                if 'firebase-test-lab-action' in uses:
                    found.add('Firebase_Compact')
                    matched.append('Firebase-Test-Lab-Action')
                if 'microsoft/appcenter-test-cli-action' in uses:
                    found.add('Appcenter')
                    matched.append('Appcenter Test CLI Action')
                if 'browserstack' in uses:
                    found.add('Browserstack')
                    matched.append('Browserstack action')
                if 'saucelabs/saucectl-run-action' in uses:
                    found.add('SauceLabs')
                    matched.append('Sauce Labs GitHub Action')

    if re.search(r'gcloud.*firebase test android run', run_text):
        found.add('Firebase_Full')
        matched.append('gcloud/firebase CLI command')

    if re.search(r'appcenter test run', run_text):
        found.add('Appcenter')
        matched.append('appcenter CLI command')

    if 'browserstack' in run_text:
        found.add('Browserstack')
        matched.append('browserstack CLI call')

    if 'saucectl' in run_text:
        found.add('SauceLabs')
        matched.append('saucectl CLI')

    return found, matched

# === MAIN PROCESSING ===
# === MAIN PROCESSING ===
results = []
for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)
            full_name = filename.split("__")[0] if "__" in filename else filename
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    raw = f.read().replace('\t', ' ')
                    ci_platform = detect_ci_platform(file_path)
                    parsed = yaml.safe_load(raw) or {}
                    run_scripts = extract_run_scripts(parsed)
                    is_unit = detect_unit_test(raw)
                    instr_types, matched_keys = detect_instrumentation_by_platform(ci_platform, raw, run_scripts, parsed)

                    # === Instrumentation Test Trigger / Device Setup ===
                    # Mark device setup if any match to DEVICE_SETUP_LABELS or *_emulator_manual
                    has_device = any(
                        label in DEVICE_SETUP_LABELS or label.endswith('_emulator_manual')
                        for label in instr_types
                    )

                    # Mark test trigger if any match TEST_TRIGGERS, Gradle, or known test commands
                    common_test_keywords = [
                        'connected', 'instrumentation', 'assembleandroidtest',
                        'runinstrumentationtests', 'am instrument', 'orchestrator', 'installdebug'
                    ]

                    has_trigger = any(
                        label in TEST_TRIGGERS
                        or 'Gradle_Command_Legacy' in label
                        for label in instr_types
                    )

                    # If manual emulator setup is found, scan run scripts for test commands
                    if not has_trigger:
                        if any(label.endswith('_emulator_manual') for label in instr_types):
                            for script in run_scripts:
                                if any(kw in script for kw in common_test_keywords):
                                    has_trigger = True
                                    break


                    results.append({
                        'filename': filename,
                        'full_name': full_name,
                        'ci_platform': ci_platform,
                        'test_type': ', '.join(sorted(instr_types)),
                        'unit_test': is_unit,
                        'instrumentation_test': bool(instr_types),
                        'Instru_T_Trigger': 'Yes' if has_trigger else 'No',
                        'Instru_T_Device_Setup': 'Yes' if has_device else 'No',
                        'matched_keywords': ', '.join(sorted(set(map(str.strip, matched_keys))))
                    })

            except Exception as e:
                results.append({
                    'filename': filename,
                    'full_name': full_name,
                    'ci_platform': 'Error',
                    'test_type': '',
                    'unit_test': False,
                    'instrumentation_test': False,
                    'Instru_T_Trigger': 'Error',
                    'Instru_T_Device_Setup': 'Error',
                    'matched_keywords': ''
                })

# === EXPORT RESULTS ===
df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ YAML scan complete! CSV saved to: {OUTPUT_CSV}")


✅ YAML scan complete! CSV saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\3.1_YML_List_ShallowC.csv
